In [1]:
import pandas as pd

In [8]:
from os.path import basename, exists


def download(url):
    filename = basename(url)
    if not exists(filename):
        from urllib.request import urlretrieve

        local, _ = urlretrieve(url, filename)
        print("Downloaded " + local)


download("https://github.com/AllenDowney/ThinkStats/raw/v3/nb/thinkstats.py")

Downloaded thinkstats.py


In [10]:
try:
    import empiricaldist
except ImportError:
    %pip install empiricaldist

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for empiricaldist: filename=empiricaldist-0.9.0-py3-none-any.whl size=14299 sha256=b362f313628da6f279f27f44bcc92ce6b33ad7902d78628aef04ac974aa2bf90
  Stored in directory: /root/.cache/pip/wheels/26/56/da/ea90b6b66dc5e72379a64e2819815066873f00c1350126e876
Successfully built empiricaldist


In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import HTML
from thinkstats import decorate

In [9]:
download("https://github.com/AllenDowney/ThinkStats/raw/v3/data/2002FemPreg.dct")
download("https://github.com/AllenDowney/ThinkStats/raw/v3/data/2002FemPreg.dat.gz")

Downloaded 2002FemPreg.dct
Downloaded 2002FemPreg.dat.gz


In [12]:
try:
    import statadict
except ImportError:
    %pip install statadict

In [13]:
dct_file = "2002FemPreg.dct"
dat_file = "2002FemPreg.dat.gz"

In [15]:
from statadict import parse_stata_dict


def read_stata(dct_file, dat_file):
    stata_dict = parse_stata_dict(dct_file)
    resp = pd.read_fwf(
        dat_file,
        names=stata_dict.names,
        colspecs=stata_dict.colspecs,
        compression="gzip",
    )
    return resp

In [19]:
preg = read_stata(dct_file, dat_file)

pandas.core.frame.DataFrame

In [21]:
preg.head(10)

,caseid,pregordr,howpreg_n,howpreg_p,moscurrp,nowprgdk,pregend1,pregend2,nbrnaliv,multbrth,...,poverty_i,laborfor_i,religion_i,metro_i,basewgt,adj_mod_basewgt,finalwgt,secu_p,sest,cmintvw
0,1,1,NaN,NaN,NaN,NaN,6.0,NaN,1.0,NaN,...,0,0,0,0,3410.389399,3869.349602,6448.271112,2,9,1231
1,1,2,NaN,NaN,NaN,NaN,6.0,NaN,1.0,NaN,...,0,0,0,0,3410.389399,3869.349602,6448.271112,2,9,1231
2,2,1,NaN,NaN,NaN,NaN,5.0,NaN,3.0,5.0,...,0,0,0,0,7226.301740,8567.549110,12999.542264,2,12,1231
3,2,2,NaN,NaN,NaN,NaN,6.0,NaN,1.0,NaN,...,0,0,0,0,7226.301740,8567.549110,12999.542264,2,12,1231
4,2,3,NaN,NaN,NaN,NaN,6.0,NaN,1.0,NaN,...,0,0,0,0,7226.301740,8567.549110,12999.542264,2,12,1231
5,6,1,NaN,NaN,NaN,NaN,6.0,NaN,1.0,NaN,...,0,0,0,0,4870.926435,5325.196999,8874.440799,1,23,1231
6,6,2,NaN,NaN,NaN,NaN,6.0,NaN,1.0,NaN,...,0,0,0,0,4870.926435,5325.196999,8874.440799,1,23,1231
7,6,3,NaN,NaN,NaN,NaN,6.0,NaN,1.0,NaN,...,0,0,0,0,4870.926435,5325.196999,8874.440799,1,23,1231
8,7,1,NaN,NaN,NaN,NaN,5.0,NaN,1.0,NaN,...,0,0,0,0,3409.579565,3787.539000,6911.879921,2,14,1233
9,7,2,NaN,NaN,NaN,NaN,5.0,NaN,1.0,NaN,...,0,0,0,0,3409.579565,3787.539000,6911.879921,2,14,1233


In [23]:
preg.shape
## so this data set contains the data of 13593 pregnancies and 243 features.

(13593, 243)

In [24]:
## let's check the feature names (columns)
preg.columns

Index(['caseid', 'pregordr', 'howpreg_n', 'howpreg_p', 'moscurrp', 'nowprgdk',
       'pregend1', 'pregend2', 'nbrnaliv', 'multbrth',
       ...
       'poverty_i', 'laborfor_i', 'religion_i', 'metro_i', 'basewgt',
       'adj_mod_basewgt', 'finalwgt', 'secu_p', 'sest', 'cmintvw'],
      dtype='object', length=243)

The NSFG dataset contains 243 variables in total. Here are some of the ones we’ll use for the explorations in this book.

caseid is the integer ID of the respondent.

pregordr is a pregnancy serial number: the code for a respondent’s first pregnancy is 1, for the second pregnancy is 2, and so on.

prglngth is the integer duration of the pregnancy in weeks.

outcome is an integer code for the outcome of the pregnancy. The code 1 indicates a live birth.

birthord is a serial number for live births: the code for a respondent’s first child is 1, and so on. For outcomes other than live birth, this field is blank.

birthwgt_lb and birthwgt_oz contain the pounds and ounces parts of the birth weight of the baby.

agepreg is the mother’s age at the end of the pregnancy.

finalwgt is the statistical weight associated with the respondent. It is a floating-point value that indicates the number of people in the U.S. population this respondent represents.

If you read the codebook carefully, you will see that many of the variables are recodes, which m

In [53]:
IQR = preg['basewgt'].quantile(.75) - preg['basewgt'].quantile(.25)
print(IQR)

## interquantile range (the difference of the 75th percentile and 25th percentile)

2534.4962143611415


In [79]:
stdbasewgt = preg['basewgt'].std()

## standard deviation

In [75]:
from statsmodels.robust.scale import mad

## to use Median absolute deviation

In [80]:
mad_basewgt = mad(preg['basewgt'])

In [82]:
print(f"La desviación estándar es: {stdbasewgt}")
print(f"La desviación absoluta mediana (MAD) es: {mad_basewgt}")
print(f"El rango intercuantílico es: {IQR}")

La desviación estándar es: 3982.6804725791208
La desviación absoluta mediana (MAD) es: 1870.0260238322933
El rango intercuantílico es: 2534.4962143611415


In [85]:
medianbasewgt = preg['basewgt'].median()
meanbasewgt = preg['basewgt'].mean()

In [86]:
print(f"La media es: {meanbasewgt}")
print(f"La desviación estándar es: {stdbasewgt}")

print(f"La mediana es: {medianbasewgt}")
print(f"La desviación absoluta mediana (MAD) es: {mad_basewgt}")


La media es: 4216.271164400659
La desviación estándar es: 3982.6804725791208
La mediana es: 3409.6485044796127
La desviación absoluta mediana (MAD) es: 1870.0260238322933
